In [ ]:
############################################Data Preparation (Data Cleaning)############################################
import pandas as pd
import os

############################################ 1 - Load the original dataset

df = pd.read_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025.csv",sep=',' , encoding="utf-8")
print(df.columns)
print(df.head(5))
print(df.tail(5))
print(df.shape)

In [ ]:
############################################ 2- Checking corrupted rows
import re

corrupted_rows = []
df['loan_status'] = df['loan_status'].astype(int)

# Business rule validations
for idx, row in df.iterrows():
    issues = []
    
    # Age validation (18-100)
    if pd.notna(row['age']) and (row['age'] < 18 or row['age'] > 100):
        issues.append('Invalid age')
    
    # Credit score validation (300-850)
    if pd.notna(row['credit_score']) and (row['credit_score'] < 300 or row['credit_score'] > 850):
        issues.append('Invalid credit_score')
    
    # Years employed validation (0-60)
    if pd.notna(row['years_employed']) and (row['years_employed'] < 0 or row['years_employed'] > 60):
        issues.append('Invalid years_employed')
    
    # Credit history years validation (0-80)
    if pd.notna(row['credit_history_years']) and (row['credit_history_years'] < 0 or row['credit_history_years'] > 80):
        issues.append('Invalid credit_history_years')
    
    # Negative values validation
    if pd.notna(row['annual_income']) and row['annual_income'] < 0:
        issues.append('Negative annual_income')
    if pd.notna(row['savings_assets']) and row['savings_assets'] < 0:
        issues.append('Negative savings_assets')
    if pd.notna(row['current_debt']) and row['current_debt'] < 0:
        issues.append('Negative current_debt')
    if pd.notna(row['loan_amount']) and row['loan_amount'] < 0:
        issues.append('Negative loan_amount')
    
    # Interest rate validation (0-100%)
    if pd.notna(row['interest_rate']) and (row['interest_rate'] < 0 or row['interest_rate'] > 100):
        issues.append('Invalid interest_rate')
    
    # Ratio validations (0-100)
    if pd.notna(row['debt_to_income_ratio']) and (row['debt_to_income_ratio'] < 0 or row['debt_to_income_ratio'] > 100):
        issues.append('Invalid debt_to_income_ratio')
    if pd.notna(row['loan_to_income_ratio']) and (row['loan_to_income_ratio'] < 0 or row['loan_to_income_ratio'] > 100):
        issues.append('Invalid loan_to_income_ratio')
    if pd.notna(row['payment_to_income_ratio']) and (row['payment_to_income_ratio'] < 0 or row['payment_to_income_ratio'] > 100):
        issues.append('Invalid payment_to_income_ratio')
    
    # Defaults and delinquencies validation (non-negative)
    if pd.notna(row['defaults_on_file']) and row['defaults_on_file'] < 0:
        issues.append('Negative defaults_on_file')
    if pd.notna(row['delinquencies_last_2yrs']) and row['delinquencies_last_2yrs'] < 0:
        issues.append('Negative delinquencies_last_2yrs')
    if pd.notna(row['derogatory_marks']) and row['derogatory_marks'] < 0:
        issues.append('Negative derogatory_marks')
    
    # String field validation using regex
    if pd.notna(row['occupation_status']):
        if not re.match(r'^[A-Za-z\s\-]+$', str(row['occupation_status'])):
            issues.append('Malformed occupation_status')
    
    if pd.notna(row['product_type']):
        if not re.match(r'^[A-Za-z\s\-]+$', str(row['product_type'])):
            issues.append('Malformed product_type')
    
    if pd.notna(row['loan_status']):
       if int(row['loan_status']) not in [0, 1]:
           issues.append('Malformed loan_status')

    
    # Logical inconsistencies
    if pd.notna(row['credit_history_years']) and pd.notna(row['age']):
        if row['credit_history_years'] > row['age'] - 18:
            issues.append('Credit history exceeds possible years')
    
    if pd.notna(row['years_employed']) and pd.notna(row['age']):
        if row['years_employed'] > row['age'] - 18:
            issues.append('Years employed exceeds possible years')
    
    # Missing critical values
    if pd.isna(row['customer_id']) or pd.isna(row['loan_status']):
        issues.append('Missing critical field')
    
    if issues:
        corrupted_rows.append({
            'index': idx,
            'customer_id': row['customer_id'],
            'issues': ', '.join(issues),
            'row_data': row
        })

# Display results
print(f"Total corrupted rows found: {len(corrupted_rows)}\n")

for corrupt in corrupted_rows:
    print(f"Row Index: {corrupt['index']}")
    print(f"Customer ID: {corrupt['customer_id']}")
    print(f"Issues: {corrupt['issues']}")
    print(f"Data: {corrupt['row_data'].to_dict()}")
    print("-" * 80)

#################### we found 144 rows that loan_to_income_ratio & debt_to_income_ratio have inproper ratio so:

# Remove rows where annual_income is missing or zero
df = df[df['annual_income'].notna()]
df = df[df['annual_income'] > 0]

# Recalculate ratios
df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income']
df['debt_to_income_ratio'] = df['current_debt'] / df['annual_income']

# Remove first row if needed
df_corrected = df.iloc[1:].copy()


# Save dataset
df_corrected.to_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025-1.csv", index=False)

print("✅ New corrected dataset saved as Loan_approval_data_2025-1.csv")

df = pd.read_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025-1.csv")

In [ ]:
############################################ 3-Checking Column Types Fix column data types
df["age"] = df["age"].astype(int)
df["defaults_on_file"] = df["defaults_on_file"].astype(int)
df["delinquencies_last_2yrs"] = df["delinquencies_last_2yrs"].astype(int)
df["derogatory_marks"] = df["derogatory_marks"].astype(int)
df["loan_status"] = df["loan_status"].astype(int)

############################################ 4-handle missing value
print(df.isna().sum(axis=1))

############################################ 5-normalize inconsistent text (“Male” vs “male”) in rows
print(df["occupation_status"].value_counts())
print(df["product_type"].value_counts())
print(df["loan_intent"].value_counts())

In [ ]:
############################################ 6- recognizing missing value
print(df.isna().sum())

############################################ 7- Preprocessing: Handle missing values if necessary
df.dropna(inplace=True)      # Example: remove rows with missing values

 ############################################ 8-Cheking Duplicated row
duplicates_count = df.duplicated().sum()
# Check if there are any duplicate rows and print the result using f-strings
if df.duplicated().any():
    print(f"Duplicates are present. Total duplicate rows: {duplicates_count}")
    df = df.drop_duplicates()
    print("Duplicates values is deleted")
else:
    print(f"No duplicates are present in the Dataset.")


In [ ]:
############################################ 7- Outlier
############################################
# 7- Outlier Analysis (Boxplot + IQR + Extreme Outliers)--> Visualizing distributions
############################################

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select numerical features to check
num_features = [
    'age',
    'years_employed',
    'annual_income',
    'credit_score',
    'credit_history_years',
    'savings_assets',
    'current_debt',
    'loan_amount',
    'interest_rate',
    'debt_to_income_ratio',
    'loan_to_income_ratio',
    'payment_to_income_ratio'
]

# -------------------------------------------------------
# 1. BOX PLOTS
# -------------------------------------------------------

plt.figure(figsize=(15, 18))
for i, col in enumerate(num_features, 1):
    plt.subplot(4, 3, i)
    sns.boxplot(data=df, y=col, color='skyblue')
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()


# -------------------------------------------------------
# 2. NORMAL & EXTREME OUTLIERS USING IQR
# -------------------------------------------------------

print("\n================ OUTLIER COUNTS (1.5*IQR and 3*IQR) ================\n")

outlier_summary_normal = {}
outlier_summary_extreme = {}

for col in num_features:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # Normal outlier bounds (1.5 * IQR)
    lower_15 = Q1 - 1.5 * IQR
    upper_15 = Q3 + 1.5 * IQR

    # Extreme outlier bounds (3 * IQR)
    lower_30 = Q1 - 3 * IQR
    upper_30 = Q3 + 3 * IQR

    # Identify normal outliers
    normal_outliers = df[(df[col] < lower_15) | (df[col] > upper_15)][col]

    # Identify extreme outliers
    extreme_outliers = df[(df[col] < lower_30) | (df[col] > upper_30)][col]

    outlier_summary_normal[col] = len(normal_outliers)
    outlier_summary_extreme[col] = len(extreme_outliers)

    print(f"{col}")
    print(f"  Normal outliers (1.5*IQR):   {len(normal_outliers)}")
    print(f"  Extreme outliers (3*IQR):   {len(extreme_outliers)}")
    print(f"  Normal bounds   : ({lower_15:.2f}, {upper_15:.2f})")
    print(f"  Extreme bounds  : ({lower_30:.2f}, {upper_30:.2f})")
    print(f"  Example extreme : {extreme_outliers.head().to_list()}")
    print("-" * 60)


# -------------------------------------------------------
# 3. SORTED SUMMARY TABLES
# -------------------------------------------------------

print("\n=============== SORTED NORMAL OUTLIER COUNTS (1.5*IQR) ===============\n")
for col, cnt in sorted(outlier_summary_normal.items(), key=lambda x: x[1], reverse=True):
    print(f"{col}: {cnt} normal outliers")

print("\n=============== SORTED EXTREME OUTLIER COUNTS (3*IQR) ===============\n")
for col, cnt in sorted(outlier_summary_extreme.items(), key=lambda x: x[1], reverse=True):
    print(f"{col}: {cnt} extreme outliers")



In [ ]:
############################################Exploratory Data Analysis (EDA)##############################
#Summarizing categorical columns
#Counting unique values per column
#Generating statistical summary for numerical data
############################################ 1-Statistical Summaries 
print(df.describe(include=['object', 'string']))
print(df.nunique())

pd.set_option('display.max_columns', None)
print(df.describe())


In [ ]:
############################################ 2-plot distributions and histograms
#Detecting skewness in numerical features
#Interpretation rule:
#Skew value	Meaning
#~0	normal distribution
#> 1	highly skewed
#< -1	left skewed

#Applying transformations (log / scaling / binary encoding):
#reduces skew
#compresses large values
#makes distribution more normal
#Log transform remove outliers indirectly

#Reducing outlier effects
#Creating a new transformed dataset (df_transform)
#Preparing features for machine learning
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
df.hist(figsize=(15,15), bins=30)
plt.tight_layout()
plt.show()

df_transform = df.copy()

print(df['years_employed'].skew())   #skew=1.29 so we use log-transform
df_transform["years_employed_log"] = np.log1p(df["years_employed"])   
#sns.histplot(df["years_employed_log"], kde=True)
#plt.show()
#print("Skew after:", df["years_employed_log"].skew()) #Skew after log-transform: -0.103
#print(df['years_employed_log'].describe()) # max=3.7 is , mean=1.69 
#Q1 = df['years_employed_log'].quantile(0.25)   # identifying outlier
#Q3 = df['years_employed_log'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['years_employed_log'] < lower_bound) | (df['years_employed_log'] > upper_bound)]
#print("Number of outliers:", len(outliers)) #Number of outliers: from 1373 to 0

print(df['current_debt'].skew())   #skew=2.43 so we use log-transform
df_transform["current_debt_log"] = np.log1p(df["current_debt"])   
#sns.histplot(df["current_debt_log"], kde=True)
#plt.show()
#print("Skew after:", df["current_debt_log"].skew()) #Skew after log-transform: -0.451
#print(df['current_debt_log'].describe()) # max=12 , mean=9
#Q1 = df['current_debt_log'].quantile(0.25)   # identifying outlier
#Q3 = df['current_debt_log'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['current_debt_log'] < lower_bound) | (df['current_debt_log'] > upper_bound)]
#print("Number of outliers:", len(outliers)) #Number of outliers: from 2932 to 638

################start-->loan_amount
from sklearn.preprocessing import StandardScaler
print(df['loan_amount'].skew())   #skew=0.93 so we use StandardScaler  
print(df['loan_amount'].describe()) # max=100000 is extremly biger than mean=33041 so for making sure i use IQR to be confident whether we have outlier or not
scaler = StandardScaler()             #StandardScaler
df_transform['loan_amount_scaled'] = scaler.fit_transform(df[['loan_amount']])
#Q1 = df['loan_amount_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['loan_amount_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['loan_amount_scaled'] < lower_bound) | (df['loan_amount_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers)) #Number of outliers: 0 so we do not need to do anything
################end-->loan_amount

print(df['annual_income'].skew())    #skew=1.88 so we use log-transform
df_transform["annual_income_log"] = np.log1p(df["annual_income"])
#sns.histplot(df["annual_income_log"], kde=True)
#plt.show()
#print("Skew after:", df["annual_income_log"].skew()) #Skew after log-transform: -0.18
#Q1 = df['annual_income_log'].quantile(0.25)   # identifying outlier
#Q3 = df['annual_income_log'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['annual_income_log'] < lower_bound) | (df['annual_income_log'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: from 2355 sto 123


print(df['credit_history_years'].skew()) #skew=0.95 so we use StandardScaler  
print(df['credit_history_years'].describe()) # max=30 is extremly biger than mean=83595 
scaler = StandardScaler()             #StandardScaler
df_transform['credit_history_years_scaled'] = scaler.fit_transform(df[['credit_history_years']])
#Q1 = df['credit_history_years_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['credit_history_years_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['credit_history_years_scaled'] < lower_bound) | (df['credit_history_years_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 349 so we need to remove them


print(df['savings_assets'].skew())                   #skew=12.1 so we use log-transform
print(df['savings_assets'].describe()) # max=300000 is extremly biger than mean=3595 so we have outlier in this feature--> removing Outlier 
df_transform["savings_assets_log"] = np.log1p(df["savings_assets"])
#sns.histplot(df["savings_assets_log"], kde=True)
#plt.show()
#print("Skew after:", df["savings_assets_log"].skew()) #Skew after log-transform: -0.24
#Q1 = df['savings_assets_log'].quantile(0.25)   # identifying outlier
#Q3 = df['savings_assets_log'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['savings_assets_log'] < lower_bound) | (df['savings_assets_log'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: from 6508 to 509

print(df['debt_to_income_ratio'].skew())   #skew=0.44 so we use StandardScaler  
print(df['debt_to_income_ratio'].describe()) # max=799 , mean=256
scaler = StandardScaler()             #StandardScaler
df_transform['debt_to_income_ratio_scaled'] = scaler.fit_transform(df[['debt_to_income_ratio']])
#Q1 = df['debt_to_income_ratio_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['debt_to_income_ratio_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['debt_to_income_ratio_scaled'] < lower_bound) | (df['debt_to_income_ratio_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 174

print(df['loan_to_income_ratio'].skew())   #skew=0.64 so we use StandardScaler  
print(df['loan_to_income_ratio'].describe()) # max=2001 , mean=622 
scaler = StandardScaler()             #StanardScalerd
df_transform['loan_to_income_ratio_scaled'] = scaler.fit_transform(df[['loan_to_income_ratio']])
#Q1 = df['loan_to_income_ratio_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['loan_to_income_ratio_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['loan_to_income_ratio_scaled'] < lower_bound) | (df['loan_to_income_ratio_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 0 

print(df['age'].skew())   #skew=0.335  
print(df['age'].describe()) #max=70 , mean=34 , median=35
scaler = StandardScaler()   #StandardScaler
df_transform['age_scaled'] = scaler.fit_transform(df[['age']])
#Q1 = df['age_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['age_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['age_scaled'] < lower_bound) | (df['age_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 112 

print(df['payment_to_income_ratio'].skew())   #skew=0.65  
print(df['payment_to_income_ratio'].describe()) #max=667 , mean=210 , median=184
scaler = StandardScaler()   #StandardScaler
df_transform['payment_to_income_ratio_scaled'] = scaler.fit_transform(df[['payment_to_income_ratio']])
#Q1 = df['payment_to_income_ratio_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['payment_to_income_ratio_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
##upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['payment_to_income_ratio_scaled'] < lower_bound) | (df['payment_to_income_ratio_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 0 

from sklearn.preprocessing import StandardScaler
print(df['loan_to_income_ratio'].skew())   #skew=0.64 
print(df['loan_to_income_ratio'].describe()) #max=2001 , mean=622 , median=547
scaler = StandardScaler()   #StandardScaler
df_transform['loan_to_income_ratio_scaled'] = scaler.fit_transform(df[['loan_to_income_ratio']])
#Q1 = df['loan_to_income_ratio_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['loan_to_income_ratio_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['loan_to_income_ratio_scaled'] < lower_bound) | (df['loan_to_income_ratio_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 0 


print(df['credit_score'].skew())   #skew=0.012 
print(df['credit_score'].describe()) #max=850 , mean=643 , median=643
scaler = StandardScaler()   #StandardScaler
df_transform['credit_score_scaled'] = scaler.fit_transform(df[['credit_score']])
#Q1 = df['credit_score_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['credit_score_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['credit_score_scaled'] < lower_bound) | (df['credit_score_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 349 

print(df['delinquencies_last_2yrs'].skew())   #skew=1.8 
print(df['delinquencies_last_2yrs'].describe()) #max=9 , mean=0.554 , median=0
#sns.histplot(df['delinquencies_last_2yrs'], kde=False, bins=10)
#plt.title('Delinquencies in the Last 2 Years - Distribution')
#plt.show()
df_transform['delinquencies_last_2yrs_binary'] = (df['delinquencies_last_2yrs'] > 0).astype(int)
#Q1 = df['delinquencies_last_2yrs_binary'].quantile(0.25)   # identifying outlier
#Q3 = df['delinquencies_last_2yrs_binary'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['delinquencies_last_2yrs_binary'] < lower_bound) | (df['delinquencies_last_2yrs_binary'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: from 1761 to 0

print(df['interest_rate'].skew())   #skew=0.0196 
print(df['interest_rate'].describe()) #max=23 , mean=15.4 , median=15.44
scaler = StandardScaler()
df_transform['interest_rate_scaled'] = scaler.fit_transform(df[['interest_rate']])
#sns.histplot(df['interest_rate'], kde=False, bins=10)
#plt.title('interest_rate - Distribution')
#plt.show()
#Q1 = df['interest_rate_scaled'].quantile(0.25)   # identifying outlier
#Q3 = df['interest_rate_scaled'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['interest_rate_scaled'] < lower_bound) | (df['interest_rate_scaled'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 0

print(df['derogatory_marks'].skew())   #skew=3.1 
print(df['derogatory_marks'].describe()) #max=4 , mean=0.147 , median=0
#sns.histplot(df['derogatory_marks'], kde=False, bins=10)
#plt.title('derogatory_marks - Distribution')
#plt.show()
df_transform['derogatory_marks_binary'] = (df['derogatory_marks'] > 0).astype(int) # because we are using descrete feature so we have to convert this feature to binary
#Q1 = df['derogatory_marks_binary'].quantile(0.25)   # identifying outlier
#Q3 = df['derogatory_marks_binary'].quantile(0.75)
#IQR = Q3 - Q1
#lower_bound = Q1 - 1.5 * IQR
#upper_bound = Q3 + 1.5 * IQR
#outliers = df[(df['derogatory_marks_binary'] < lower_bound) | (df['derogatory_marks_binary'] > upper_bound)]
#print("Number of outliers:", len(outliers))   #Number of outliers: 0--at binary feature having outlier is not make sense because we have just 0 and 1
df_transform.columns

In [ ]:
###############################################################################3-correlation analysis
########correlation analysis with oroginal features without scaled and binary and log tranformed features
#Selecting only numeric original features
#Computing Spearman correlation
#Measuring relationship with loan_status
#Visualizing feature importance
#Plotting full correlation heatmap

num_df = df.select_dtypes(include=[np.number]).copy()
corr_spearman = num_df.corr(method='spearman')
#Spearman is better than Pearson because:
#✔ works for non-linear relationships
#✔ robust to outliers
#✔ works well with skewed financial data

# Focus on correlation with the target
target_corr = corr_spearman['loan_status'].sort_values(ascending=False)
# Visualize this subset
plt.figure(figsize=(6, 8))
sns.barplot(x=target_corr.values, y=target_corr.index, hue=target_corr.index, palette='viridis', legend=False)
plt.title("Feature Correlation with Loan Status before tranformed features")
plt.show()
print("Correlation Matrix-corr_spearman:")
print(corr_spearman)

mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr_spearman, mask=mask, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Cleaned Correlation Heatmap befor tranformed features")
plt.show()

In [ ]:
###############################################################################3-correlation analysis
########correlation analysis with scaled and binary and log tranformed features

# -----------------------------
# Build transformed feature set
# -----------------------------
df_work = df.copy()

# Safe log transforms (handle zeros)
for col in ["current_debt", "annual_income", "savings_assets"]:
    if col in df_work.columns:
        df_work[f"{col}_log"] = np.log1p(df_work[col])

# Scale continuous features (StandardScaler-like without fitting state)
def standardize(series: pd.Series):
    s = series.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-12)

scale_cols = {
    "loan_amount": "loan_amount_scaled",
    "credit_history_years": "credit_history_years_scaled",
    "debt_to_income_ratio": "debt_to_income_ratio_scaled",
    "age": "age_scaled",
    "payment_to_income_ratio": "payment_to_income_ratio_scaled",
    "loan_to_income_ratio": "loan_to_income_ratio_scaled",
    "credit_score": "credit_score_scaled",
    "interest_rate": "interest_rate_scaled",
}
for src, dst in scale_cols.items():
    if src in df_work.columns:
        df_work[dst] = standardize(df_work[src])

# Binary features
if "delinquencies_last_2yrs" in df_work.columns:
    df_work["delinquencies_last_2yrs_binary"] = (df_work["delinquencies_last_2yrs"].fillna(0) > 0).astype(int)
if "derogatory_marks" in df_work.columns:
    df_work["derogatory_marks_binary"] = (df_work["derogatory_marks"].fillna(0) > 0).astype(int)

# Years employed log
if "years_employed" in df_work.columns:
    df_work["years_employed_log"] = np.log1p(df_work["years_employed"])

# Keep defaults_on_file as is (assumed binary/integer)
# Ensure loan_status exists and is numeric/binary
assert "loan_status" in df_work.columns, "loan_status column is required"

# ---------------------------------------------------
# Select only the requested engineered feature columns
# ---------------------------------------------------
requested_features = [
    "age_scaled",
    "years_employed_log",
    "annual_income_log",
    "credit_score_scaled",
    "credit_history_years_scaled",
    "savings_assets_log",
    "current_debt_log",
    "defaults_on_file",
    "delinquencies_last_2yrs_binary",
    "derogatory_marks_binary",
    "loan_amount_scaled",
    "interest_rate_scaled",
    "debt_to_income_ratio_scaled",
    "loan_to_income_ratio_scaled", 
    "payment_to_income_ratio_scaled",
    
    "loan_status",  # include target for correlation
]
available_features = [f for f in requested_features if f in df_work.columns]
df_feat = df_work[available_features].copy()

# --------------------------------
# Spearman correlation and plotting
# --------------------------------
num_df = df_feat.select_dtypes(include=[np.number]).copy()
corr_spearman = num_df.corr(method='spearman')

target_corr = corr_spearman['loan_status'].drop(labels=['loan_status']).sort_values(ascending=False)

plt.figure(figsize=(7, max(4, 0.4 * len(target_corr))))
sns.barplot(
    x=target_corr.values,
    y=target_corr.index,
    hue=target_corr.index,
    palette='viridis',
    legend=False
)
plt.title("Feature Spearman Correlation with Loan Status after tranformed features")
plt.xlabel("Spearman ρ")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("Spearman Correlation Matrix (selected engineered features after tranformed features):")
print(corr_spearman)

mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr_spearman, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Spearman ρ'})
plt.title("Engineered Feature Correlation Heatmap (Spearman) after tranformed features")
plt.tight_layout()
plt.show()

In [ ]:
##############################################4-class imbalance Checking
# Assume target column is 'loan_status'
#How many samples belong to each class in loan_status
#Whether the dataset is balanced or biased
#Whether you need special handling (like weighting or resampling)

class_counts = df['loan_status'].value_counts()
class_percent = df['loan_status'].value_counts(normalize=True) * 100
print("Class counts:")
print(class_counts)
print("\nClass percentages:")
print(class_percent)

# Bar plot
sns.countplot(x='loan_status', data=df)
plt.title('Class Distribution')
plt.xlabel('Classes')
plt.ylabel('Count')
plt.show()

df['loan_status'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Class Distribution')
plt.ylabel('')
plt.show()

majority_class_count = class_counts.max()
minority_class_count = class_counts.min()
imbalance_ratio = majority_class_count / minority_class_count
print(f'Imbalance Ratio: {imbalance_ratio:.2f}') ############IR is 1.2 so we need to use class weighting

In [ ]:
############################################ 5-Modality with transforemd features
#Plotting distributions of transformed features
#Checking whether they are:
#unimodal (one peak)
#bimodal (two peaks)
#multimodal (multiple peaks)

# List of the features you want to plot
features = [
    'years_employed_log',
    'current_debt_log',
    'loan_amount_scaled',
    'annual_income_log',
    'credit_history_years_scaled',
    'savings_assets_log',
    'age_scaled',
    'credit_score_scaled',
    'interest_rate_scaled',
    'age_scaled',
]

# Plot each feature
for col in features:
    if col in df_transform.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(df_transform[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Column '{col}' not found in dataframe.")#among our features , only loan_amount is Multimodality so we have to categorize it to three diffrent groups


In [ ]:
#########recognizing important features##################################### 1- XGBoost to select importance features for original features
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# -------------------------------------------------------
# 1. Select only engineered + one-hot features
# -------------------------------------------------------
selected_features = [
    'years_employed',
    'current_debt',
    'loan_amount',
    'annual_income',
    'credit_history_years',
    'savings_assets',
    'debt_to_income_ratio',
    'loan_to_income_ratio',
    'age',
    'payment_to_income_ratio',
    'credit_score',
    'interest_rate',
    'delinquencies_last_2yrs',
    'derogatory_marks',
    'defaults_on_file',
    
]

X = df[selected_features]
y = df["loan_status"]

# -------------------------------------------------------
# 2. Train/test split
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------
# 3. Train XGBoost model
# -------------------------------------------------------
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

# -------------------------------------------------------
# 4. Feature Importance Extraction
# -------------------------------------------------------
importance_values = model.feature_importances_
feature_names = X.columns

importance_series = pd.Series(importance_values, index=feature_names)
importance_series = importance_series.sort_values(ascending=False)

print("\n=== XGBoost Feature Importance (sorted) ===")
print(importance_series)

# -------------------------------------------------------
# 5. Plot Feature Importance
# -------------------------------------------------------
plt.figure(figsize=(14, 7))
importance_series.plot(kind='bar')
plt.title("XGBoost Feature Importance (Engineered Features Only)")
plt.ylabel("Importance Score")
plt.xlabel("Feature Name")
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 6. Identify weak features
# -------------------------------------------------------
threshold = 0.02
low_importance = importance_series[importance_series < threshold]

print("\n=== Features recommended for removal (importance < 0.01) ===")
print(low_importance)

In [ ]:
#########recognizing important features##################################### 1- XGBoost to select importance features for Transformed features
import matplotlib.pyplot as plt 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# -------------------------------------------------------
# 1. Select only engineered + one-hot features
# -------------------------------------------------------
selected_features = [
    'years_employed_log', 'current_debt_log', 'loan_amount_scaled',
       'annual_income_log', 'credit_history_years_scaled',
       'savings_assets_log', 'debt_to_income_ratio_scaled',
       'loan_to_income_ratio_scaled', 'age_scaled',
       'payment_to_income_ratio_scaled', 'credit_score_scaled',
       'delinquencies_last_2yrs_binary', 'interest_rate_scaled',
       'derogatory_marks_binary', 
    
]

X = df_transform[selected_features]
y = df_transform["loan_status"]

# -------------------------------------------------------
# 2. Train/test split
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------
# 3. Train XGBoost model
# -------------------------------------------------------
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

# -------------------------------------------------------
# 4. Feature Importance Extraction
# -------------------------------------------------------
importance_values = model.feature_importances_
feature_names = X.columns

importance_series = pd.Series(importance_values, index=feature_names)
importance_series = importance_series.sort_values(ascending=False)

print("\n=== XGBoost Feature Importance (sorted) ===")
print(importance_series)

# -------------------------------------------------------
# 5. Plot Feature Importance
# -------------------------------------------------------
plt.figure(figsize=(14, 7))
importance_series.plot(kind='bar')
plt.title("XGBoost Feature Importance (Engineered Features Only)")
plt.ylabel("Importance Score")
plt.xlabel("Feature Name")
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 6. Identify weak features
# -------------------------------------------------------
threshold = 0.02
low_importance = importance_series[importance_series < threshold]

print("\n=== Features recommended for removal (importance < 0.01) ===")
print(low_importance)

In [ ]:
##################################### 2- Permutation Importance to select importance features
#############################################
# 2- Permutation Importance (using selected_features)
#############################################

#Taking your engineered feature set
#Using a trained model (model)
#Measuring how performance changes when each feature is shuffled
#Ranking features based on true impact on prediction

#Concept:
#For each feature:
#Step:
#Shuffle one feature column
#Keep all others unchanged
#Measure drop in model performance
#Repeat multiple times

#interpretation:
#Change     after shuffle	    Meaning
#big drop-->                 important feature
#smal drop-->                 	weak feature
#no change-->	               useless feature

from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

# Ensure X and y use your selected engineered features
selected_features = [
    'years_employed_log', 'current_debt_log', 'loan_amount_scaled',
       'annual_income_log', 'credit_history_years_scaled',
       'savings_assets_log', 'debt_to_income_ratio_scaled',
       'loan_to_income_ratio_scaled', 'age_scaled',
       'payment_to_income_ratio_scaled', 'credit_score_scaled',
       'delinquencies_last_2yrs_binary', 'interest_rate_scaled',
       'derogatory_marks_binary', 
    
]

X = df_transform[selected_features]
y = df_transform["loan_status"]


# Calculate permutation importance
perm_result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

perm_importances = pd.Series(
    perm_result.importances_mean,
    index=X.columns
).sort_values(ascending=False)

print("\n=== Permutation Importance ===")
print(perm_importances)

# Plot
plt.figure(figsize=(12, 4))
perm_importances.plot(kind='bar')
plt.title("Permutation Importance (Using Selected Features)")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

In [ ]:
##################################### 3- SHAP Values to select importance features
#############################################
# SHAP Values for Selected Features
#############################################
#Select engineered features
#Build SHAP TreeExplainer
#Compute SHAP values on test data
#Plot global importance
#Rank features by contribution

import shap
import pandas as pd

# Use your selected engineered features
selected_features = [
    'years_employed_log', 'current_debt_log', 'loan_amount_scaled',
       'annual_income_log', 'credit_history_years_scaled',
       'savings_assets_log', 'debt_to_income_ratio_scaled',
       'loan_to_income_ratio_scaled', 'age_scaled',
       'payment_to_income_ratio_scaled', 'credit_score_scaled',
       'delinquencies_last_2yrs_binary', 'interest_rate_scaled',
       'derogatory_marks_binary', 
    
]

X = df_transform[selected_features]
y = df_transform["loan_status"]
# SHAP Explainer
explainer = shap.TreeExplainer(model)

# SHAP Values on the same X_test used for evaluation
shap_values = explainer.shap_values(X_test)

print("Generating SHAP plots...")

# Summary scatter plot
shap.summary_plot(shap_values, X_test)

# Summary bar chart
shap.summary_plot(shap_values, X_test, plot_type="bar")

###################################################
# PRINT GLOBAL SHAP IMPORTANCE VALUES (sorted)
###################################################
shap_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": np.abs(shap_values).mean(axis=0)
}).sort_values("importance", ascending=False)

print("\n=== SHAP Global Feature Importance ===")
print(shap_importance) # we will remove the features like savings_assets_log /
                         #occupation_status_Employed/occupation_status_Self-Employed/occupation_status_Student

In [ ]:
#
# A loan approval prediction system that:
#predicts default risk (loan_status)
#handles imbalance
#optimizes business cost (FN vs FP)
#tunes hyperparameters
#evaluates with AUC + confusion matrix
#finds best threshold (NOT just 0.5)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Numerical features to apply log transform
log_features = ['current_debt', 'annual_income', 'savings_assets']

# Numerical features to scale
scale_features = ['loan_amount', 'credit_history_years', 'debt_to_income_ratio', 
                  'loan_to_income_ratio', 'age', 'payment_to_income_ratio', 
                  'credit_score', 'interest_rate']

# Binary features
binary_features = ['delinquencies_last_2yrs', 'derogatory_marks']

# Categorical features
categorical_features = ['occupation_status', 'product_type', 'loan_intent']

# ---------------------------------------------------------
# Step 1: Train/Validation/Test Split
# ---------------------------------------------------------
X = df.drop(columns=['loan_status', 'customer_id'])
y = df['loan_status']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Prevent data leakage - use .copy()
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# ---------------------------------------------------------
# Step 2: Log transform (only on positive values)
# ---------------------------------------------------------
for col in log_features:
    X_train[f'{col}_log'] = np.log1p(X_train[col])
    X_val[f'{col}_log'] = np.log1p(X_val[col])
    X_test[f'{col}_log'] = np.log1p(X_test[col])

# ---------------------------------------------------------
# Step 3: Scaling — fit only on train
# ---------------------------------------------------------
scalers = {}

for col in scale_features:
    scaler = StandardScaler()
    X_train[f'{col}_scaled'] = scaler.fit_transform(X_train[[col]])
    scalers[col] = scaler
    X_val[f'{col}_scaled'] = scaler.transform(X_val[[col]])
    X_test[f'{col}_scaled'] = scaler.transform(X_test[[col]])

log_scaled_features = [f'{col}_log' for col in log_features]

for col in log_scaled_features:
    scaler = StandardScaler()
    X_train[f'{col}_scaled'] = scaler.fit_transform(X_train[[col]])
    scalers[col] = scaler
    X_val[f'{col}_scaled'] = scaler.transform(X_val[[col]])
    X_test[f'{col}_scaled'] = scaler.transform(X_test[[col]])

# ---------------------------------------------------------
# Step 4: Convert binary features
# ---------------------------------------------------------
for col in binary_features:
    X_train[f'{col}_binary'] = (X_train[col] > 0).astype(int)
    X_val[f'{col}_binary'] = (X_val[col] > 0).astype(int)
    X_test[f'{col}_binary'] = (X_test[col] > 0).astype(int)

# ---------------------------------------------------------
# Step 5: One-Hot Encoding
# ---------------------------------------------------------
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
X_val_encoded = pd.get_dummies(X_val, columns=categorical_features, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Drop original features after transformation
features_to_drop = log_features + scale_features + binary_features
X_train_encoded = X_train_encoded.drop(columns=features_to_drop, errors='ignore')
X_val_encoded = X_val_encoded.drop(columns=features_to_drop, errors='ignore')
X_test_encoded = X_test_encoded.drop(columns=features_to_drop, errors='ignore')

# ---------------------------------------------------------
# Step 6: Final feature selection
# ---------------------------------------------------------
final_features = [
    'credit_score_scaled',
    'debt_to_income_ratio_scaled',
    'age_scaled',
    'credit_history_years_scaled',
    'defaults_on_file',
    'annual_income_log_scaled',
    'current_debt_log_scaled',
    'loan_amount_scaled',
    'delinquencies_last_2yrs_binary',
    'derogatory_marks_binary',
    'interest_rate_scaled',
    'payment_to_income_ratio_scaled',
    'loan_to_income_ratio_scaled',
    'savings_assets_log_scaled'
]

categorical_cols = [col for col in X_train_encoded.columns 
                   if any(col.startswith(cat + '_') for cat in categorical_features)]
final_features += categorical_cols

final_features = [f for f in final_features if f in X_train_encoded.columns]

X_train_final = X_train_encoded[final_features]
X_val_final = X_val_encoded[final_features]
X_test_final = X_test_encoded[final_features]

print("="*80)
print("DATASET SUMMARY")
print("="*80)
print(f"Train: {X_train_final.shape}, Val: {X_val_final.shape}, Test: {X_test_final.shape}")
print(f"Features: {len(final_features)}")
print(f"Class distribution - Train: {dict(y_train.value_counts())}")
print(f"Class distribution - Val: {dict(y_val.value_counts())}")
print(f"Class distribution - Test: {dict(y_test.value_counts())}")

# ---------------------------------------------------------
# ############################################################################Compute class weights
# ---------------------------------------------------------
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
sample_weights = np.array([class_weights[i] for i in y_train])

print(f"\nClass weights: [0: {class_weights[0]:.4f}, 1: {class_weights[1]:.4f}]")

# ---------------------------------------------------------
######################################################################### HYPERPARAMETER GRID SEARCH
# ---------------------------------------------------------
print("\n" + "="*80)
print("STARTING HYPERPARAMETER GRID SEARCH")
print("="*80)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [200, 300, 400, 500,600,700],
    'learning_rate': [0.03, 0.05, 0.07, 0.1 ,0.2, 0.3],
    'max_depth': [3, 4, 5, 6,7,8],
    'min_child_weight': [1, 3, 5, 7, 9]
}

# Fixed parameters
fixed_params = {
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42
}

# Generate all combinations
param_combinations = list(product(
    param_grid['n_estimators'],
    param_grid['learning_rate'],
    param_grid['max_depth'],
    param_grid['min_child_weight']
))

print(f"Total combinations to test: {len(param_combinations)}")
print(f"This will take approximately {len(param_combinations) * 0.5:.1f}-{len(param_combinations) * 1:.1f} minutes\n")

# Store results
results = []

# Test each combination
for idx, (n_est, lr, max_d, min_cw) in enumerate(param_combinations, 1):
    print(f"[{idx}/{len(param_combinations)}] Testing: n_estimators={n_est}, lr={lr}, max_depth={max_d}, min_child_weight={min_cw}")
    
    # Create model
    model = XGBClassifier(
        n_estimators=n_est,
        learning_rate=lr,
        max_depth=max_d,
        min_child_weight=min_cw,
        early_stopping_rounds=30,
        **fixed_params
    )
    
    # Train model
    model.fit(
        X_train_final, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_val_final, y_val)],
        verbose=0
    )
    
    # Predictions
    y_val_pred_proba = model.predict_proba(X_val_final)[:, 1]
    
    # Calculate AUC
    val_auc = roc_auc_score(y_val, y_val_pred_proba)
    
    ################################################################## Find optimal threshold - CORRECTED
    thresholds = np.arange(0.15, 0.6, 0.01)
    best_score = np.inf  # FIXED: Initialize to infinity for cost minimization
    best_thresh = 0.5
    best_fn = 0
    best_fp = 0
    
    for thresh in thresholds:
        y_val_pred = (y_val_pred_proba >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred).ravel()
        
        ######################### Cost function: prioritize low FN, penalize FP
        cost = 2.5 * fn + 1.0 * fp
        
        # Calculate ratio
        if fn > 0:
            fp_fn_ratio = fp / fn
        else:
            fp_fn_ratio = fp
        
        # Penalty for severe imbalance
        if fp_fn_ratio < 3:
            balance_penalty = 100
        elif fp_fn_ratio > 20:
            balance_penalty = 50
        else:
            balance_penalty = 0
        
        total_score = cost + balance_penalty
        
        # FIXED: Correct comparison for minimization
        if total_score < best_score:
            best_score = total_score
            best_thresh = thresh
            best_fn = fn
            best_fp = fp
    
    # Store results
    results.append({
        'n_estimators': n_est,
        'learning_rate': lr,
        'max_depth': max_d,
        'min_child_weight': min_cw,
        'val_auc': val_auc,
        'best_threshold': best_thresh,
        'fn': best_fn,
        'fp': best_fp,
        'fp_fn_ratio': best_fp / best_fn if best_fn > 0 else best_fp,
        'total_errors': best_fn + best_fp,
        'balance_score': best_score
    })
    
    print(f"  → Val AUC: {val_auc:.4f}, Threshold: {best_thresh:.2f}, FN: {best_fn}, FP: {best_fp}, Ratio: {best_fp/best_fn if best_fn > 0 else best_fp:.1f}")

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by multiple criteria - CORRECTED
# Higher AUC is better → rank descending (ascending=False)
# Lower errors is better → rank ascending (ascending=True)
# Lower balance_score is better → rank ascending (ascending=True)
results_df['auc_rank'] = results_df['val_auc'].rank(ascending=False)  # FIXED: Higher AUC = rank 1
results_df['error_rank'] = results_df['total_errors'].rank(ascending=True)
results_df['balance_rank'] = results_df['balance_score'].rank(ascending=True)

# Composite score: lower is better (since all ranks are 1-based, lower rank = better)
results_df['composite_score'] = (
    0.4 * results_df['auc_rank'] + 
    0.3 * results_df['error_rank'] + 
    0.3 * results_df['balance_rank']
)

results_df = results_df.sort_values('composite_score')

print("\n" + "="*80)
print("TOP 10 HYPERPARAMETER COMBINATIONS")
print("="*80)
print(results_df.head(10)[['n_estimators', 'learning_rate', 'max_depth', 'min_child_weight', 
                            'val_auc', 'best_threshold', 'fn', 'fp', 'fp_fn_ratio', 'total_errors']].to_string(index=False))

# Get best parameters
best_params = results_df.iloc[0]

print("\n" + "="*80)
print("BEST HYPERPARAMETERS (Composite Score)")
print("="*80)
print(f"n_estimators: {int(best_params['n_estimators'])}")
print(f"learning_rate: {best_params['learning_rate']}")
print(f"max_depth: {int(best_params['max_depth'])}")
print(f"min_child_weight: {int(best_params['min_child_weight'])}")
print(f"Validation AUC: {best_params['val_auc']:.4f}")
print(f"Optimal Threshold: {best_params['best_threshold']:.2f}")
print(f"False Negatives: {int(best_params['fn'])}")
print(f"False Positives: {int(best_params['fp'])}")
print(f"FP/FN Ratio: {best_params['fp_fn_ratio']:.2f}")

# ---------------------------------------------------------
# TRAIN FINAL MODEL WITH BEST PARAMETERS
# ---------------------------------------------------------
print("\n" + "="*80)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*80)

final_model = XGBClassifier(
    n_estimators=int(best_params['n_estimators']),
    learning_rate=best_params['learning_rate'],
    max_depth=int(best_params['max_depth']),
    min_child_weight=int(best_params['min_child_weight']),
    early_stopping_rounds=30,
    **fixed_params
)

final_model.fit(
    X_train_final, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train_final, y_train), (X_val_final, y_val)],
    verbose=50
)

# ---------------------------------------------------------
# FINAL EVALUATION ON TEST SET
# ---------------------------------------------------------
y_train_pred_proba = final_model.predict_proba(X_train_final)[:, 1]
y_val_pred_proba = final_model.predict_proba(X_val_final)[:, 1]
y_test_pred_proba = final_model.predict_proba(X_test_final)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred_proba)
val_auc = roc_auc_score(y_val, y_val_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print("\n" + "="*80)
print("FINAL MODEL AUC SCORES")
print("="*80)
print(f"Train AUC: {train_auc:.4f}")
print(f"Val AUC: {val_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Train-Val Gap: {abs(train_auc - val_auc):.4f}")

# Apply best threshold on test set
best_threshold = best_params['best_threshold']
y_test_pred = (y_test_pred_proba >= best_threshold).astype(int)

print("\n" + "="*80)
print(f"TEST SET PERFORMANCE (Threshold={best_threshold:.2f})")
print("="*80)
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nTN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print(f"FP/FN Ratio: {fp/fn if fn > 0 else fp:.2f}")
print(f"Total Errors: {fp + fn}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

# Calculate costs with different ratios
print("\n" + "="*80)
print("COST ANALYSIS")
print("="*80)
for fn_weight in [2, 3, 4, 5]:
    cost = fn_weight * fn + 1 * fp
    print(f"Cost (FN weight={fn_weight}:1): {cost}")

# ---------------------------------------------------------
# VISUALIZATIONS
# ---------------------------------------------------------

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Test Set (Optimized Model)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.savefig('roc_curve_optimized.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nROC curve saved: roc_curve_optimized.png")

# 2. Feature Importance
feature_importance = pd.DataFrame({
    'feature': final_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances (Optimized Model)', fontsize=14, fontweight='bold')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance_optimized.png', dpi=300, bbox_inches='tight')
plt.close()
print("Feature importance plot saved: feature_importance_optimized.png")

# 3. Hyperparameter Search Results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# AUC by n_estimators
axes[0, 0].scatter(results_df['n_estimators'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[0, 0].set_xlabel('n_estimators')
axes[0, 0].set_ylabel('Validation AUC')
axes[0, 0].set_title('AUC vs n_estimators')
axes[0, 0].grid(True, alpha=0.3)

# AUC by learning_rate
axes[0, 1].scatter(results_df['learning_rate'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[0, 1].set_xlabel('learning_rate')
axes[0, 1].set_ylabel('Validation AUC')
axes[0, 1].set_title('AUC vs learning_rate')
axes[0, 1].grid(True, alpha=0.3)

# AUC by max_depth
axes[1, 0].scatter(results_df['max_depth'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[1, 0].set_xlabel('max_depth')
axes[1, 0].set_ylabel('Validation AUC')
axes[1, 0].set_title('AUC vs max_depth')
axes[1, 0].grid(True, alpha=0.3)

# FN vs FP scatter
scatter = axes[1, 1].scatter(results_df['fp'], results_df['fn'], alpha=0.6, c=results_df['val_auc'], cmap='viridis', s=50)
axes[1, 1].scatter(best_params['fp'], best_params['fn'], color='red', s=200, marker='*', 
                   edgecolors='black', linewidths=2, label='Best Model', zorder=5)
axes[1, 1].set_xlabel('False Positives')
axes[1, 1].set_ylabel('False Negatives')
axes[1, 1].set_title('FP vs FN Trade-off')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 1], label='Validation AUC')

plt.tight_layout()
plt.savefig('hyperparameter_search_results.png', dpi=300, bbox_inches='tight')
plt.close()
print("Hyperparameter search visualization saved: hyperparameter_search_results.png")

# 4. Top 10 Features
print("\n" + "="*80)
print("TOP 10 MOST IMPORTANT FEATURES")
print("="*80)
print(feature_importance.head(10).to_string(index=False))

# Save results to CSV
results_df.to_csv('hyperparameter_search_results.csv', index=False)
print("\nFull hyperparameter search results saved: hyperparameter_search_results.csv")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)
print(f"Best model achieves {test_auc:.4f} AUC on test set")
print(f"With only {fn} false negatives and {fp} false positives")
print(f"FP/FN ratio: {fp/fn if fn > 0 else fp:.2f} (balanced)")


In [ ]:
# leakage control
#2. business cost function
# 3. threshold tuning
# 4. hyperparameter search logic
# 5. evaluation philosophy


#1. Train / Validation / Test Split
#Prevents overfitting
#Prevents data leakage during model tuning
#🔹 2. Realistic Cost Function
#cost = 2.5 * FN + 1.0 * FP
#Reflects real business impact
#False Negatives are more expensive than False Positives
#Important for domains like credit risk (loan default prediction)
#🔹 3. Multi-objective Model Selection
#The best model is not selected only based on AUC
#Multiple factors are considered:
#False Positives (FP)
#False Negatives (FN)
#Class balance
#AUC score
#The goal is to optimize real-world performance, not just accuracy
#🔹 4. Smart Threshold Tuning
#Threshold is not fixed at 0.5
#It is tested across a range (e.g., 0.15 to 0.6)
#The optimal threshold is selected based on:
#FN minimization
#FP control
#Business cost function
#This is important because the default 0.5 threshold is often suboptimal
#🔹 5. More Advanced Hyperparameter Search
#Uses validation set (eval_set)
#Includes early stopping
#Uses validation-based model selection
#Uses composite scoring instead of single metric
#🔹 6. Multi-objective Decision Making
#The best model is not chosen based only on AUC
#Instead, multiple criteria are considered:
#AUC score
#False Positives
#False Negatives
#Class balance
#This leads to a more realistic and business-oriented model selection process

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import shap
import warnings
warnings.filterwarnings('ignore')

# Numerical features to apply log transform
log_features = ['current_debt', 'annual_income', 'savings_assets']

# Numerical features to scale
scale_features = ['loan_amount', 'credit_history_years', 'debt_to_income_ratio', 
                  'loan_to_income_ratio', 'age', 'payment_to_income_ratio', 
                  'credit_score', 'interest_rate']

# Binary features
binary_features = ['delinquencies_last_2yrs', 'derogatory_marks']

# Categorical features
categorical_features = ['occupation_status', 'product_type', 'loan_intent']

# ---------------------------------------------------------
# Step 1: Train/Validation/Test Split
# ---------------------------------------------------------
X = df.drop(columns=['loan_status', 'customer_id'])
y = df['loan_status']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Prevent data leakage - use .copy()
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# ---------------------------------------------------------
# Step 2: Log transform (only on positive values)
# ---------------------------------------------------------
for col in log_features:
    X_train[f'{col}_log'] = np.log1p(X_train[col])
    X_val[f'{col}_log'] = np.log1p(X_val[col])
    X_test[f'{col}_log'] = np.log1p(X_test[col])

# ---------------------------------------------------------
# Step 3: Scaling — fit only on train
# ---------------------------------------------------------
scalers = {}

for col in scale_features:
    scaler = StandardScaler()
    X_train[f'{col}_scaled'] = scaler.fit_transform(X_train[[col]])
    scalers[col] = scaler
    X_val[f'{col}_scaled'] = scaler.transform(X_val[[col]])
    X_test[f'{col}_scaled'] = scaler.transform(X_test[[col]])

log_scaled_features = [f'{col}_log' for col in log_features]

for col in log_scaled_features:
    scaler = StandardScaler()
    X_train[f'{col}_scaled'] = scaler.fit_transform(X_train[[col]])
    scalers[col] = scaler
    X_val[f'{col}_scaled'] = scaler.transform(X_val[[col]])
    X_test[f'{col}_scaled'] = scaler.transform(X_test[[col]])

# ---------------------------------------------------------
# Step 4: Convert binary features
# ---------------------------------------------------------
for col in binary_features:
    X_train[f'{col}_binary'] = (X_train[col] > 0).astype(int)
    X_val[f'{col}_binary'] = (X_val[col] > 0).astype(int)
    X_test[f'{col}_binary'] = (X_test[col] > 0).astype(int)

# ---------------------------------------------------------
# Step 5: One-Hot Encoding
# ---------------------------------------------------------
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
X_val_encoded = pd.get_dummies(X_val, columns=categorical_features, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Drop original features after transformation
features_to_drop = log_features + scale_features + binary_features
X_train_encoded = X_train_encoded.drop(columns=features_to_drop, errors='ignore')
X_val_encoded = X_val_encoded.drop(columns=features_to_drop, errors='ignore')
X_test_encoded = X_test_encoded.drop(columns=features_to_drop, errors='ignore')

# ---------------------------------------------------------
# Step 6: Final feature selection
# ---------------------------------------------------------
final_features = [
    'credit_score_scaled',
    'debt_to_income_ratio_scaled',
    'age_scaled',
    'credit_history_years_scaled',
    'defaults_on_file',
    'annual_income_log_scaled',
    'current_debt_log_scaled',
    'loan_amount_scaled',
    'delinquencies_last_2yrs_binary',
    'derogatory_marks_binary',
    'interest_rate_scaled',
    'payment_to_income_ratio_scaled',
    'loan_to_income_ratio_scaled',
    'savings_assets_log_scaled'
]

categorical_cols = [col for col in X_train_encoded.columns 
                   if any(col.startswith(cat + '_') for cat in categorical_features)]
final_features += categorical_cols

final_features = [f for f in final_features if f in X_train_encoded.columns]

X_train_final = X_train_encoded[final_features]
X_val_final = X_val_encoded[final_features]
X_test_final = X_test_encoded[final_features]

print("="*80)
print("DATASET SUMMARY")
print("="*80)
print(f"Train: {X_train_final.shape}, Val: {X_val_final.shape}, Test: {X_test_final.shape}")
print(f"Features: {len(final_features)}")
print(f"Class distribution - Train: {dict(y_train.value_counts())}")
print(f"Class distribution - Val: {dict(y_val.value_counts())}")
print(f"Class distribution - Test: {dict(y_test.value_counts())}")

# ---------------------------------------------------------
# Compute class weights
# ---------------------------------------------------------
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
sample_weights = np.array([class_weights[i] for i in y_train])

print(f"\nClass weights: [0: {class_weights[0]:.4f}, 1: {class_weights[1]:.4f}]")

# ---------------------------------------------------------
# HYPERPARAMETER GRID SEARCH
# ---------------------------------------------------------
print("\n" + "="*80)
print("STARTING HYPERPARAMETER GRID SEARCH")
print("="*80)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [200, 300, 400, 500],
    'learning_rate': [0.03, 0.05, 0.07, 0.1],
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 3, 5, 7]
}

# Fixed parameters
fixed_params = {
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42
}

# Generate all combinations
param_combinations = list(product(
    param_grid['n_estimators'],
    param_grid['learning_rate'],
    param_grid['max_depth'],
    param_grid['min_child_weight']
))

print(f"Total combinations to test: {len(param_combinations)}")
print(f"This will take approximately {len(param_combinations) * 0.5:.1f}-{len(param_combinations) * 1:.1f} minutes\n")

# Store results
results = []

# Test each combination
for idx, (n_est, lr, max_d, min_cw) in enumerate(param_combinations, 1):
    print(f"[{idx}/{len(param_combinations)}] Testing: n_estimators={n_est}, lr={lr}, max_depth={max_d}, min_child_weight={min_cw}")
    
    # Create model
    model = XGBClassifier(
        n_estimators=n_est,
        learning_rate=lr,
        max_depth=max_d,
        min_child_weight=min_cw,
        early_stopping_rounds=30,
        **fixed_params
    )
    
    # Train model
    model.fit(
        X_train_final, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_val_final, y_val)],
        verbose=0
    )
    
    # Predictions
    y_val_pred_proba = model.predict_proba(X_val_final)[:, 1]
    
    # Calculate AUC
    val_auc = roc_auc_score(y_val, y_val_pred_proba)
    
    # Find optimal threshold
    thresholds = np.arange(0.15, 0.6, 0.01)
    best_score = np.inf
    best_thresh = 0.5
    best_fn = 0
    best_fp = 0
    
    for thresh in thresholds:
        y_val_pred = (y_val_pred_proba >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred).ravel()
        
        # Cost function: prioritize low FN, penalize FP
        cost = 2.5 * fn + 1.0 * fp
        
        # Calculate ratio
        if fn > 0:
            fp_fn_ratio = fp / fn
        else:
            fp_fn_ratio = fp
        
        # Penalty for severe imbalance
        if fp_fn_ratio < 3:
            balance_penalty = 100
        elif fp_fn_ratio > 20:
            balance_penalty = 50
        else:
            balance_penalty = 0
        
        total_score = cost + balance_penalty
        
        if total_score < best_score:
            best_score = total_score
            best_thresh = thresh
            best_fn = fn
            best_fp = fp
    
    # Store results
    results.append({
        'n_estimators': n_est,
        'learning_rate': lr,
        'max_depth': max_d,
        'min_child_weight': min_cw,
        'val_auc': val_auc,
        'best_threshold': best_thresh,
        'fn': best_fn,
        'fp': best_fp,
        'fp_fn_ratio': best_fp / best_fn if best_fn > 0 else best_fp,
        'total_errors': best_fn + best_fp,
        'balance_score': best_score
    })
    
    print(f"  → Val AUC: {val_auc:.4f}, Threshold: {best_thresh:.2f}, FN: {best_fn}, FP: {best_fp}, Ratio: {best_fp/best_fn if best_fn > 0 else best_fp:.1f}")

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by multiple criteria
results_df['auc_rank'] = results_df['val_auc'].rank(ascending=False)
results_df['error_rank'] = results_df['total_errors'].rank(ascending=True)
results_df['balance_rank'] = results_df['balance_score'].rank(ascending=True)

# Composite score: lower is better
results_df['composite_score'] = (
    0.4 * results_df['auc_rank'] + 
    0.3 * results_df['error_rank'] + 
    0.3 * results_df['balance_rank']
)

results_df = results_df.sort_values('composite_score')

print("\n" + "="*80)
print("TOP 10 HYPERPARAMETER COMBINATIONS")
print("="*80)
print(results_df.head(10)[['n_estimators', 'learning_rate', 'max_depth', 'min_child_weight', 
                            'val_auc', 'best_threshold', 'fn', 'fp', 'fp_fn_ratio', 'total_errors']].to_string(index=False))

# Get best parameters
best_params = results_df.iloc[0]

print("\n" + "="*80)
print("BEST HYPERPARAMETERS (Composite Score)")
print("="*80)
print(f"n_estimators: {int(best_params['n_estimators'])}")
print(f"learning_rate: {best_params['learning_rate']}")
print(f"max_depth: {int(best_params['max_depth'])}")
print(f"min_child_weight: {int(best_params['min_child_weight'])}")
print(f"Validation AUC: {best_params['val_auc']:.4f}")
print(f"Optimal Threshold: {best_params['best_threshold']:.2f}")
print(f"False Negatives: {int(best_params['fn'])}")
print(f"False Positives: {int(best_params['fp'])}")
print(f"FP/FN Ratio: {best_params['fp_fn_ratio']:.2f}")

# ---------------------------------------------------------
# TRAIN FINAL MODEL WITH BEST PARAMETERS
# ---------------------------------------------------------
print("\n" + "="*80)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("="*80)

final_model = XGBClassifier(
    n_estimators=int(best_params['n_estimators']),
    learning_rate=best_params['learning_rate'],
    max_depth=int(best_params['max_depth']),
    min_child_weight=int(best_params['min_child_weight']),
    early_stopping_rounds=30,
    **fixed_params
)

final_model.fit(
    X_train_final, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train_final, y_train), (X_val_final, y_val)],
    verbose=50
)

# ---------------------------------------------------------
# FINAL EVALUATION ON TEST SET
# ---------------------------------------------------------
y_train_pred_proba = final_model.predict_proba(X_train_final)[:, 1]
y_val_pred_proba = final_model.predict_proba(X_val_final)[:, 1]
y_test_pred_proba = final_model.predict_proba(X_test_final)[:, 1]

train_auc = roc_auc_score(y_train, y_train_pred_proba)
val_auc = roc_auc_score(y_val, y_val_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print("\n" + "="*80)
print("FINAL MODEL AUC SCORES")
print("="*80)
print(f"Train AUC: {train_auc:.4f}")
print(f"Val AUC: {val_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Train-Val Gap: {abs(train_auc - val_auc):.4f}")

# Apply best threshold on test set
best_threshold = best_params['best_threshold']
y_test_pred = (y_test_pred_proba >= best_threshold).astype(int)

print("\n" + "="*80)
print(f"TEST SET PERFORMANCE (Threshold={best_threshold:.2f})")
print("="*80)
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nTN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print(f"FP/FN Ratio: {fp/fn if fn > 0 else fp:.2f}")
print(f"Total Errors: {fp + fn}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

# Calculate costs with different ratios
print("\n" + "="*80)
print("COST ANALYSIS")
print("="*80)
for fn_weight in [2, 3, 4, 5]:
    cost = fn_weight * fn + 1 * fp
    print(f"Cost (FN weight={fn_weight}:1): {cost}")

# ---------------------------------------------------------
# VISUALIZATIONS (DISPLAY IN JUPYTER NOTEBOOK)
# ---------------------------------------------------------

# 1. ROC Curve
print("\n" + "="*80)
print("ROC CURVE")
print("="*80)
fpr, tpr, _ = roc_curve(y_test, y_test_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Test Set (Optimized Model)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Feature Importance
print("\n" + "="*80)
print("FEATURE IMPORTANCE")
print("="*80)
feature_importance = pd.DataFrame({
    'feature': final_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances (Optimized Model)', fontsize=14, fontweight='bold')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

# 3. Hyperparameter Search Results
print("\n" + "="*80)
print("HYPERPARAMETER SEARCH VISUALIZATION")
print("="*80)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# AUC by n_estimators
axes[0, 0].scatter(results_df['n_estimators'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[0, 0].set_xlabel('n_estimators')
axes[0, 0].set_ylabel('Validation AUC')
axes[0, 0].set_title('AUC vs n_estimators')
axes[0, 0].grid(True, alpha=0.3)

# AUC by learning_rate
axes[0, 1].scatter(results_df['learning_rate'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[0, 1].set_xlabel('learning_rate')
axes[0, 1].set_ylabel('Validation AUC')
axes[0, 1].set_title('AUC vs learning_rate')
axes[0, 1].grid(True, alpha=0.3)

# AUC by max_depth
axes[1, 0].scatter(results_df['max_depth'], results_df['val_auc'], alpha=0.6, c=results_df['val_auc'], cmap='viridis')
axes[1, 0].set_xlabel('max_depth')
axes[1, 0].set_ylabel('Validation AUC')
axes[1, 0].set_title('AUC vs max_depth')
axes[1, 0].grid(True, alpha=0.3)

# FN vs FP scatter
scatter = axes[1, 1].scatter(results_df['fp'], results_df['fn'], alpha=0.6, c=results_df['val_auc'], cmap='viridis', s=50)
axes[1, 1].scatter(best_params['fp'], best_params['fn'], color='red', s=200, marker='*', 
                   edgecolors='black', linewidths=2, label='Best Model', zorder=5)
axes[1, 1].set_xlabel('False Positives')
axes[1, 1].set_ylabel('False Negatives')
axes[1, 1].set_title('FP vs FN Trade-off')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 1], label='Validation AUC')

plt.tight_layout()
plt.show()

# 4. Top 10 Features
print("\n" + "="*80)
print("TOP 10 MOST IMPORTANT FEATURES")
print("="*80)
print(feature_importance.head(10).to_string(index=False))

# Save results to CSV (optional)
results_df.to_csv('hyperparameter_search_results.csv', index=False)
print("\nFull hyperparameter search results saved: hyperparameter_search_results.csv")


In [ ]:
# =========================================================================Explainable AI system
# SHAP ANALYSIS - COMPREHENSIVE MODEL INTERPRETATION
# =========================================================================
print("\n" + "="*80)
print("STARTING SHAP ANALYSIS")
print("="*80)
print("This will take a few minutes to compute SHAP values...")

# Create SHAP explainer for tree-based model
explainer = shap.TreeExplainer(final_model)

# Calculate SHAP values on test set (using a sample for speed if dataset is large)
if len(X_test_final) > 2000:
    print(f"Using sample of 2000 instances for SHAP analysis (from {len(X_test_final)} total)")
    shap_sample_indices = np.random.choice(len(X_test_final), 2000, replace=False)
    X_shap = X_test_final.iloc[shap_sample_indices]
    y_shap = y_test.iloc[shap_sample_indices]
    y_test_pred_proba_shap = y_test_pred_proba[shap_sample_indices]
else:
    X_shap = X_test_final
    y_shap = y_test
    y_test_pred_proba_shap = y_test_pred_proba

print(f"Computing SHAP values for {len(X_shap)} samples...")
shap_values = explainer.shap_values(X_shap)

print("SHAP computation complete!")

# ---------------------------------------------------------
# SHAP Visualization 1: Summary Plot (Beeswarm)
# ---------------------------------------------------------
print("\n" + "="*80)
print("SHAP SUMMARY PLOT (BEESWARM)")
print("="*80)
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, feature_names=final_features, show=False, max_display=20)
plt.title('SHAP Summary Plot - Feature Impact on Model Output', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# SHAP Visualization 2: Summary Plot (Bar - Mean Absolute SHAP)
# ---------------------------------------------------------
print("\n" + "="*80)
print("SHAP FEATURE IMPORTANCE (BAR PLOT)")
print("="*80)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=final_features, plot_type="bar", show=False, max_display=20)
plt.title('SHAP Feature Importance - Mean |SHAP Value|', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# SHAP Analysis: Calculate Mean Absolute SHAP Values
# ---------------------------------------------------------
print("\n" + "="*80)
print("SHAP-BASED FEATURE IMPORTANCE RANKING")
print("="*80)

# Calculate mean absolute SHAP value for each feature
shap_importance = pd.DataFrame({
    'feature': final_features,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
    'mean_shap': shap_values.mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

# Calculate percentage contribution
shap_importance['percentage'] = (shap_importance['mean_abs_shap'] / shap_importance['mean_abs_shap'].sum()) * 100

# Add cumulative percentage
shap_importance['cumulative_percentage'] = shap_importance['percentage'].cumsum()

print("\nTop 20 Features by SHAP Importance:")
display(shap_importance.head(20)[['feature', 'mean_abs_shap', 'mean_shap', 'percentage', 'cumulative_percentage']])

# ---------------------------------------------------------
# SHAP Visualization 3: Waterfall Plots (Individual Predictions)
# ---------------------------------------------------------
print("\n" + "="*80)
print("WATERFALL PLOTS FOR SPECIFIC PREDICTIONS")
print("="*80)

# Convert predictions to binary using best threshold
y_shap_pred = (y_test_pred_proba_shap >= best_threshold).astype(int)

# Find specific prediction types
tp_indices = np.where((y_shap == 1) & (y_shap_pred == 1))[0]
tn_indices = np.where((y_shap == 0) & (y_shap_pred == 0))[0]
fp_indices = np.where((y_shap == 0) & (y_shap_pred == 1))[0]
fn_indices = np.where((y_shap == 1) & (y_shap_pred == 0))[0]

# Function to create waterfall plot
def create_waterfall_plot_jupyter(idx, title):
    if len(idx) > 0:
        sample_idx = idx[0]
        print(f"\n{title}")
        print("-" * 80)
        
        plt.figure(figsize=(12, 8))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[sample_idx],
                base_values=explainer.expected_value,
                data=X_shap.iloc[sample_idx].values,
                feature_names=final_features
            ),
            max_display=15,
            show=False
        )
        plt.title(title, fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        # Print prediction details
        actual = y_shap.iloc[sample_idx]
        predicted = y_shap_pred[sample_idx]
        prob = y_test_pred_proba_shap[sample_idx]
        print(f"Actual: {actual}, Predicted: {predicted}, Probability: {prob:.4f}")
    else:
        print(f"\n{title}")
        print("-" * 80)
        print("No samples found for this category")

# Create waterfall plots for each prediction type
create_waterfall_plot_jupyter(
    tp_indices,
    'SHAP Waterfall - True Positive (Correctly Predicted Default)'
)

create_waterfall_plot_jupyter(
    tn_indices,
    'SHAP Waterfall - True Negative (Correctly Predicted Non-Default)'
)

create_waterfall_plot_jupyter(
    fn_indices,
    'SHAP Waterfall - False Negative (Missed Default)'
)

create_waterfall_plot_jupyter(
    fp_indices,
    'SHAP Waterfall - False Positive (Incorrectly Predicted Default)'
)

# ---------------------------------------------------------
# SHAP Visualization 4: Dependence Plots (Top Features)
# ---------------------------------------------------------
print("\n" + "="*80)
print("DEPENDENCE PLOTS FOR TOP 5 FEATURES")
print("="*80)

top_5_features = shap_importance.head(5)['feature'].tolist()

for i, feature in enumerate(top_5_features, 1):
    feature_idx = final_features.index(feature)
    print(f"\n[{i}/5] Dependence Plot: {feature}")
    print("-" * 80)
    
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        feature_idx,
        shap_values,
        X_shap,
        feature_names=final_features,
        show=False,
        interaction_index='auto'
    )
    plt.title(f'SHAP Dependence Plot - {feature}', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# SHAP Visualization 5: Force Plot (Interactive)
# ---------------------------------------------------------
print("\n" + "="*80)
print("INTERACTIVE FORCE PLOT")
print("="*80)

# Create force plot for first 100 predictions (or all if less than 100)
n_samples_force = min(100, len(X_shap))
print(f"Displaying interactive force plot for {n_samples_force} samples...")

# Display force plot inline in Jupyter
shap.initjs()
display(shap.force_plot(
    explainer.expected_value,
    shap_values[:n_samples_force],
    X_shap.iloc[:n_samples_force],
    feature_names=final_features
))

# ---------------------------------------------------------
# SHAP Analysis: Interaction Effects (if dataset is small enough)
# ---------------------------------------------------------
if len(X_shap) <= 500:
    print("\n" + "="*80)
    print("SHAP INTERACTION ANALYSIS")
    print("="*80)
    print("Computing interaction values (this may take several minutes)...")
    
    try:
        shap_interaction_values = explainer.shap_interaction_values(X_shap)
        
        # Get top 2 features for interaction analysis
        top_2_features = shap_importance.head(2)['feature'].tolist()
        feature_idx_1 = final_features.index(top_2_features[0])
        feature_idx_2 = final_features.index(top_2_features[1])
        
        print(f"\nInteraction between: {top_2_features[0]} × {top_2_features[1]}")
        print("-" * 80)
        
        plt.figure(figsize=(10, 8))
        shap.dependence_plot(
            (feature_idx_1, feature_idx_2),
            shap_interaction_values,
            X_shap,
            feature_names=final_features,
            show=False
        )
        plt.title(f'SHAP Interaction: {top_2_features[0]} × {top_2_features[1]}', 
                  fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"⚠ Interaction analysis skipped: {str(e)}")
else:
    print("\n" + "="*80)
    print("SHAP INTERACTION ANALYSIS SKIPPED")
    print("="*80)
    print(f"Dataset too large ({len(X_shap)} samples). Interaction analysis requires ≤500 samples.")

# ---------------------------------------------------------
# SHAP Summary Statistics
# ---------------------------------------------------------
print("\n" + "="*80)
print("SHAP ANALYSIS SUMMARY")
print("="*80)

print(f"\nTotal features analyzed: {len(final_features)}")
print(f"Samples used for SHAP: {len(X_shap)}")
print(f"\nTop 5 features by mean |SHAP|:")
for idx, row in shap_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['mean_abs_shap']:.4f} ({row['percentage']:.2f}%)")

print(f"\nCumulative importance of top 10 features: {shap_importance.head(10)['percentage'].sum():.2f}%")
print(f"Cumulative importance of top 20 features: {shap_importance.head(20)['percentage'].sum():.2f}%")

print("\n" + "="*80)
print("SHAP ANALYSIS COMPLETE")
print("="*80)



In [ ]:
################################################################ RELIABILITY MONITORING 
#Drift Detection
#↓بعد
#Reliability Monitoring
# =========================================================================
print("\n" + "="*80)
print("RELIABILITY MONITORING & SELECTIVE PREDICTION SETUP")
print("="*80)

# ---------------------------------------------------------
# PART 1: ESTABLISH REFERENCE BASELINE
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 1: ESTABLISHING REFERENCE BASELINE")
print("="*80)

# Use validation set as reference (can also combine train+val)
X_reference = X_val_final.copy()
y_reference = y_val.copy()
y_reference_pred_proba = final_model.predict_proba(X_reference)[:, 1]

# Calculate reference statistics for each feature
reference_stats = {}

for feature in final_features:
    feature_data = X_reference[feature]
    
    # Check if feature is numeric (not boolean)
    is_numeric = feature_data.dtype in ['float64', 'int64', 'float32', 'int32']
    is_binary = feature_data.nunique() <= 2
    
    if is_numeric and not is_binary:
        # Full statistics for continuous features
        reference_stats[feature] = {
            'mean': feature_data.mean(),
            'std': feature_data.std(),
            'min': feature_data.min(),
            'max': feature_data.max(),
            'q25': feature_data.quantile(0.25),
            'q50': feature_data.quantile(0.50),
            'q75': feature_data.quantile(0.75),
            'q90': feature_data.quantile(0.90),
            'q95': feature_data.quantile(0.95)
        }
    else:
        # Limited statistics for binary/categorical features
        reference_stats[feature] = {
            'mean': feature_data.mean() if is_numeric else None,
            'std': feature_data.std() if is_numeric else None,
            'min': feature_data.min() if is_numeric else None,
            'max': feature_data.max() if is_numeric else None,
            'unique_values': feature_data.nunique(),
            'mode': feature_data.mode()[0] if len(feature_data.mode()) > 0 else None
        }

# Reference prediction statistics
reference_pred_stats = {
    'mean_prob': y_reference_pred_proba.mean(),
    'std_prob': y_reference_pred_proba.std(),
    'median_prob': np.median(y_reference_pred_proba),
    'approval_rate': (y_reference_pred_proba < best_threshold).mean(),
    'rejection_rate': (y_reference_pred_proba >= best_threshold).mean()
}

# Reference performance metrics
y_reference_pred = (y_reference_pred_proba >= best_threshold).astype(int)
cm_ref = confusion_matrix(y_reference, y_reference_pred)
tn_ref, fp_ref, fn_ref, tp_ref = cm_ref.ravel()

reference_performance = {
    'auc': roc_auc_score(y_reference, y_reference_pred_proba),
    'tn': tn_ref,
    'fp': fp_ref,
    'fn': fn_ref,
    'tp': tp_ref,
    'fp_fn_ratio': fp_ref / fn_ref if fn_ref > 0 else fp_ref,
    'total_errors': fp_ref + fn_ref,
    'cost': 2.5 * fn_ref + 1.0 * fp_ref,
    'threshold': best_threshold
}

print("\nReference Baseline Summary:")
print("-" * 80)
print(f"Reference samples: {len(X_reference)}")
print(f"Reference AUC: {reference_performance['auc']:.4f}")
print(f"Reference FN: {reference_performance['fn']}, FP: {reference_performance['fp']}")
print(f"Reference FP/FN Ratio: {reference_performance['fp_fn_ratio']:.2f}")
print(f"Reference Cost: {reference_performance['cost']:.2f}")
print(f"Reference Mean Probability: {reference_pred_stats['mean_prob']:.4f}")
print(f"Reference Approval Rate: {reference_pred_stats['approval_rate']:.2%}")

# ---------------------------------------------------------
## # # # # # # # # # # #  PART 2: PSI (POPULATION STABILITY INDEX) CALCULATION
# has the distribution of the new data changed compared with the training data?
print("\n" + "="*80)
print("PART 2: PSI CALCULATION FUNCTION")
print("="*80)

def calculate_psi(reference_data, new_data, bins=10):
    """
    Calculate Population Stability Index (PSI)
    
    PSI < 0.1: No significant change
    0.1 <= PSI < 0.2: Moderate change (monitor)
    PSI >= 0.2: Significant change (investigate/retrain)
    """
    # Create bins based on reference data
    if reference_data.nunique() <= 10:  # Categorical or low cardinality
        # For categorical features
        ref_counts = reference_data.value_counts(normalize=True, dropna=False)
        new_counts = new_data.value_counts(normalize=True, dropna=False)
        
        # Align indices
        all_categories = set(ref_counts.index) | set(new_counts.index)
        ref_pct = pd.Series({cat: ref_counts.get(cat, 0.0001) for cat in all_categories})
        new_pct = pd.Series({cat: new_counts.get(cat, 0.0001) for cat in all_categories})
    else:
        # For continuous features
        breakpoints = np.percentile(reference_data, np.linspace(0, 100, bins + 1))
        breakpoints = np.unique(breakpoints)  # Remove duplicates
        
        ref_binned = pd.cut(reference_data, bins=breakpoints, include_lowest=True, duplicates='drop')
        new_binned = pd.cut(new_data, bins=breakpoints, include_lowest=True, duplicates='drop')
        
        ref_pct = ref_binned.value_counts(normalize=True, dropna=False)
        new_pct = new_binned.value_counts(normalize=True, dropna=False)
        
        # Align indices
        all_bins = set(ref_pct.index) | set(new_pct.index)
        ref_pct = pd.Series({b: ref_pct.get(b, 0.0001) for b in all_bins})
        new_pct = pd.Series({b: new_pct.get(b, 0.0001) for b in all_bins})
    
    # Replace zeros with small value to avoid log(0)
    ref_pct = ref_pct.replace(0, 0.0001)
    new_pct = new_pct.replace(0, 0.0001)
    
    # Calculate PSI
    psi = np.sum((new_pct - ref_pct) * np.log(new_pct / ref_pct))
    
    return psi

print("PSI function defined successfully")
print("\nPSI Interpretation:")
print("  PSI < 0.1  : No significant change")
print("  0.1 ≤ PSI < 0.2 : Moderate change (monitor)")
print("  PSI ≥ 0.2  : Significant change (investigate/retrain)")
print("  PSI ≥ 0.3  : Severe drift (immediate action required)")

# ---------------------------------------------------------
# # # # # # # # # # # # # # # # # # # # # # # # # PART 3: KS STATISTIC CALCULATION
# Are the two statistical distributions different from each other?
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 3: KS STATISTIC FUNCTION")
print("="*80)

from scipy.stats import ks_2samp

def calculate_ks_statistic(reference_data, new_data):
    """
    Calculate Kolmogorov-Smirnov statistic
    
    KS < 0.05: No significant difference
    0.05 <= KS < 0.1: Moderate difference
    KS >= 0.1: Significant difference
    """
    # Remove NaN values
    ref_clean = reference_data.dropna()
    new_clean = new_data.dropna()
    
    if len(ref_clean) == 0 or len(new_clean) == 0:
        return 0.0, 1.0
    
    # For categorical features, convert to numeric codes
    if reference_data.dtype == 'object' or reference_data.dtype.name == 'category':
        all_categories = list(set(ref_clean.unique()) | set(new_clean.unique()))
        cat_to_num = {cat: i for i, cat in enumerate(all_categories)}
        ref_clean = ref_clean.map(cat_to_num)
        new_clean = new_clean.map(cat_to_num)
    
    ks_stat, p_value = ks_2samp(ref_clean, new_clean)
    return ks_stat, p_value

print("KS statistic function defined successfully")
print("\nKS Interpretation:")
print("  KS < 0.05  : No significant difference")
print("  0.05 ≤ KS < 0.1 : Moderate difference (monitor)")
print("  KS ≥ 0.1   : Significant difference (investigate)")

# ---------------------------------------------------------
# PART 4: COMPREHENSIVE DRIFT MONITORING FUNCTION
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 4: DRIFT MONITORING FUNCTION")
print("="*80)

def monitor_data_drift(X_new, X_reference, feature_names, 
                       psi_threshold_moderate=0.1, 
                       psi_threshold_severe=0.2,
                       ks_threshold=0.1):
    """
    Monitor data drift across all features
    """
    drift_results = []
    
    for feature in feature_names:
        ref_data = X_reference[feature]
        new_data = X_new[feature]
        
        # Calculate PSI
        psi = calculate_psi(ref_data, new_data)
        
        # Calculate KS
        ks_stat, ks_pvalue = calculate_ks_statistic(ref_data, new_data)
        
        # Determine drift status
        if psi >= psi_threshold_severe or ks_stat >= ks_threshold:
            status = 'SEVERE'
        elif psi >= psi_threshold_moderate:
            status = 'MODERATE'
        else:
            status = 'STABLE'
        
        # Calculate mean only for numeric features
        is_numeric = ref_data.dtype in ['float64', 'int64', 'float32', 'int32']
        
        drift_results.append({
            'feature': feature,
            'psi': psi,
            'ks_statistic': ks_stat,
            'ks_pvalue': ks_pvalue,
            'status': status,
            'mean_ref': ref_data.mean() if is_numeric else None,
            'mean_new': new_data.mean() if is_numeric else None
        })
    
    drift_df = pd.DataFrame(drift_results).sort_values('psi', ascending=False)
    return drift_df

print("Drift monitoring function defined successfully")

# ---------------------------------------------------------
# PART 5: PREDICTION DRIFT MONITORING
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 5: PREDICTION DRIFT MONITORING FUNCTION")
print("="*80)

def monitor_prediction_drift(y_new_proba, y_reference_proba, 
                            threshold, 
                            prob_diff_threshold=0.05,
                            approval_rate_diff_threshold=0.10):
    """
    Monitor drift in model predictions
    """
    # Calculate statistics for new predictions
    new_stats = {
        'mean_prob': y_new_proba.mean(),
        'std_prob': y_new_proba.std(),
        'median_prob': np.median(y_new_proba),
        'approval_rate': (y_new_proba < threshold).mean(),
        'rejection_rate': (y_new_proba >= threshold).mean()
    }
    
    # Calculate statistics for reference predictions
    ref_stats = {
        'mean_prob': y_reference_proba.mean(),
        'std_prob': y_reference_proba.std(),
        'median_prob': np.median(y_reference_proba),
        'approval_rate': (y_reference_proba < threshold).mean(),
        'rejection_rate': (y_reference_proba >= threshold).mean()
    }
    
    # Calculate differences
    prob_diff = abs(new_stats['mean_prob'] - ref_stats['mean_prob'])
    approval_rate_diff = abs(new_stats['approval_rate'] - ref_stats['approval_rate'])
    
    # Determine drift status
    alerts = []
    if prob_diff > prob_diff_threshold:
        alerts.append(f"Mean probability shifted by {prob_diff:.4f}")
    if approval_rate_diff > approval_rate_diff_threshold:
        alerts.append(f"Approval rate changed by {approval_rate_diff:.2%}")
    
    # Calculate PSI for prediction distribution
    pred_psi = calculate_psi(
        pd.Series(y_reference_proba),
        pd.Series(y_new_proba),
        bins=10
    )
    
    if pred_psi > 0.2:
        alerts.append(f"Prediction PSI = {pred_psi:.4f} (SEVERE)")
    elif pred_psi > 0.1:
        alerts.append(f"Prediction PSI = {pred_psi:.4f} (MODERATE)")
    
    return {
        'new_stats': new_stats,
        'ref_stats': ref_stats,
        'prob_diff': prob_diff,
        'approval_rate_diff': approval_rate_diff,
        'pred_psi': pred_psi,
        'alerts': alerts,
        'drift_detected': len(alerts) > 0
    }

print("Prediction drift monitoring function defined successfully")

# ---------------------------------------------------------
# PART 6: PERFORMANCE MONITORING (WITH LABELS)
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 6: PERFORMANCE MONITORING FUNCTION")
print("="*80)

def monitor_performance(y_true, y_pred_proba, threshold,
                       reference_performance,
                       auc_threshold=0.95,
                       fn_multiplier=1.5):
    """
    Monitor model performance when true labels are available
    """
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate metrics
    auc = roc_auc_score(y_true, y_pred_proba)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    fp_fn_ratio = fp / fn if fn > 0 else fp
    total_errors = fp + fn
    
    # Calculate cost (using 2.5x weight for FN as in grid search)
    cost = 2.5 * fn + 1.0 * fp
    
    # Compare with reference
    alerts = []
    
    if auc < auc_threshold:
        alerts.append(f"AUC dropped to {auc:.4f} (threshold: {auc_threshold})")
    
    if auc < reference_performance['auc'] - 0.02:
        alerts.append(f"AUC decreased by {reference_performance['auc'] - auc:.4f}")
    
    if fn > reference_performance['fn'] * fn_multiplier:
        alerts.append(f"FN increased to {fn} (reference: {reference_performance['fn']})")
    
    if fp_fn_ratio < 3:
        alerts.append(f"FP/FN ratio too low: {fp_fn_ratio:.2f} (FN dominates)")
    
    if total_errors > reference_performance['total_errors'] * 1.3:
        alerts.append(f"Total errors increased by {((total_errors/reference_performance['total_errors'])-1)*100:.1f}%")
    
    return {
        'auc': auc,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'fp_fn_ratio': fp_fn_ratio,
        'total_errors': total_errors,
        'cost': cost,
        'alerts': alerts,
        'performance_degraded': len(alerts) > 0
    }

print("Performance monitoring function defined successfully")

print("\n" + "="*80)
print("RELIABILITY MONITORING FRAMEWORK SETUP COMPLETE")
print("="*80)


In [ ]:
# =========================================================================
# PART 13: PRODUCTION-STYLE MONITORING ORCHESTRATION
# =========================================================================
import json
import os
from datetime import datetime

MONITORING_LOG_PATH = "monitoring_history.jsonl"   # append-only log, like a real monitoring DB
BASELINE_PATH = "reference_baseline.json"           # persisted baseline so it survives restarts

# ---------------------------------------------------------
# 1. Persist the reference baseline to disk (not just in memory)
# ---------------------------------------------------------
def save_baseline(reference_stats, reference_pred_stats, reference_performance, path=BASELINE_PATH):
    """
    In production the baseline can't just live in a notebook variable —
    the monitoring job runs as a separate process/schedule, so it needs
    the baseline saved somewhere it can reload from.
    """
    baseline = {
        "created_at": datetime.now().isoformat(),
        "reference_stats": {k: {sk: (float(sv) if sv is not None else None)
                                  for sk, sv in v.items()}
                             for k, v in reference_stats.items()},
        "reference_pred_stats": {k: float(v) for k, v in reference_pred_stats.items()},
        "reference_performance": {k: float(v) for k, v in reference_performance.items()},
    }
    with open(path, "w") as f:
        json.dump(baseline, f, indent=2)
    print(f"Baseline saved to {path}")

def load_baseline(path=BASELINE_PATH):
    with open(path, "r") as f:
        return json.load(f)

# ---------------------------------------------------------
# 2. A single orchestration function — this is "the monitoring job"
#    that would actually be scheduled (e.g. Airflow DAG, cron job,
#    Databricks job) to run automatically on new incoming data
# ---------------------------------------------------------
def run_monitoring_job(X_new, y_new, model, X_reference, y_reference,
                        y_reference_pred_proba, final_features,
                        best_threshold, reference_performance,
                        batch_id=None):
    """
    Simulates what a scheduled monitoring job does end-to-end:
    1. score the new batch
    2. run all three monitoring checks
    3. persist the result with a timestamp
    4. decide if action is required
    """
    timestamp = datetime.now().isoformat()
    batch_id = batch_id or timestamp

    y_new_pred_proba = model.predict_proba(X_new)[:, 1]

    drift_report = monitor_data_drift(X_new, X_reference, final_features)
    pred_drift_report = monitor_prediction_drift(y_new_pred_proba, y_reference_pred_proba, best_threshold)

    has_labels = y_new is not None
    if has_labels:
        perf_report = monitor_performance(y_new, y_new_pred_proba, best_threshold, reference_performance)
    else:
        # In real production, labels (actual defaults) often only arrive weeks/months later
        # (you don't know someone defaulted until they actually miss payments)
        perf_report = None

    severe_drift = drift_report[drift_report['status'] == 'SEVERE']

    action_required = (
        len(severe_drift) > 0 or
        pred_drift_report['drift_detected'] or
        (perf_report is not None and perf_report['performance_degraded'])
    )

    record = {
        "batch_id": batch_id,
        "timestamp": timestamp,
        "n_samples": len(X_new),
        "severe_drift_features": severe_drift['feature'].tolist(),
        "prediction_psi": pred_drift_report['pred_psi'],
        "prediction_drift_detected": pred_drift_report['drift_detected'],
        "mean_predicted_prob": float(y_new_pred_proba.mean()),
        "approval_rate": float((y_new_pred_proba < best_threshold).mean()),
        "labels_available": has_labels,
        "auc": perf_report['auc'] if perf_report else None,
        "fn": int(perf_report['fn']) if perf_report else None,
        "fp": int(perf_report['fp']) if perf_report else None,
        "action_required": bool(action_required),
    }

    # Append to the monitoring log — this is what a dashboard would read from
    with open(MONITORING_LOG_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

    if action_required:
        send_alert(record, reference_performance)

    return record

# ---------------------------------------------------------
# 3. Alerting — in real systems this posts to Slack/email/PagerDuty,
#    here we simulate it, but the interface is what matters
# ---------------------------------------------------------
def send_alert(record, reference_performance):
    """
    Stub for real alerting. Swap the print() for:
      - requests.post(slack_webhook_url, json={...})
      - boto3 SNS publish
      - send_email(...)
    """
    print("\n" + "🚨"*20)
    print(f"ALERT — batch {record['batch_id']} at {record['timestamp']}")
    if record['severe_drift_features']:
        print(f"  Severe drift in: {record['severe_drift_features']}")
    if record['prediction_drift_detected']:
        print(f"  Prediction PSI: {record['prediction_psi']:.4f}")
    if record['auc'] is not None and record['auc'] < reference_performance['auc'] - 0.02:
        print(f"  AUC dropped to {record['auc']:.4f}")
    print("🚨"*20 + "\n")

# ---------------------------------------------------------
# 4. Retraining trigger — codifies "when do we automatically retrain?"
# ---------------------------------------------------------
def check_retraining_trigger(log_path=MONITORING_LOG_PATH, lookback=5):
    """
    A common real-world rule: if the last N monitoring runs show
    action_required = True, or 2+ consecutive severe drifts,
    flag the model for retraining instead of relying on a human
    to notice.
    """
    if not os.path.exists(log_path):
        return False, "No monitoring history yet"

    with open(log_path, "r") as f:
        records = [json.loads(line) for line in f.readlines()[-lookback:]]

    if len(records) < 2:
        return False, "Not enough history yet"

    consecutive_action_required = sum(r['action_required'] for r in records)

    if consecutive_action_required >= lookback - 1:
        return True, f"{consecutive_action_required}/{lookback} recent runs required action"

    return False, "Model stable, no retraining needed"

# ---------------------------------------------------------
# 5. DEMO: simulate several monitoring runs over "time"
#    (splitting the test set into batches, as if data arrived daily)
# ---------------------------------------------------------
print("\n" + "="*80)
print("SIMULATING SCHEDULED MONITORING OVER MULTIPLE BATCHES")
print("="*80)

save_baseline(reference_stats, reference_pred_stats, reference_performance)

n_batches = 4
batch_size = len(X_test_final) // n_batches

for i in range(n_batches):
    start, end = i * batch_size, (i + 1) * batch_size
    X_batch = X_test_final.iloc[start:end]
    y_batch = y_test.iloc[start:end]

    record = run_monitoring_job(
        X_batch, y_batch, final_model,
        X_reference, y_reference, y_reference_pred_proba,
        final_features, best_threshold, reference_performance,
        batch_id=f"batch_{i+1}"
    )
    print(f"Batch {i+1}: AUC={record['auc']:.4f}, action_required={record['action_required']}")

should_retrain, reason = check_retraining_trigger()
print(f"\nRetraining trigger: {should_retrain} — {reason}")

In [ ]:
# ---------------------------------------------------------SELECTIVE PREDICTION FRAMEWORK
# PART 7: SELECTIVE PREDICTION FRAMEWORK
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 7: SELECTIVE PREDICTION FRAMEWORK")
print("="*80)

def selective_prediction(y_proba, t1, t2):
    """
    Implement selective prediction with three zones:
    - Zone 1 (p < t1): Confident ACCEPT (low risk)
    - Zone 2 (t1 <= p <= t2): UNCERTAIN (manual review)
    - Zone 3 (p > t2): Confident REJECT (high risk)
    
    Parameters:
    -----------
    y_proba : array
        Predicted probabilities of default
    t1 : float
        Lower threshold (accept if below)
    t2 : float
        Upper threshold (reject if above)
    
    Returns:
    --------
    decisions : array
        0 = Accept, 1 = Reject, 2 = Manual Review
    """
    decisions = np.zeros(len(y_proba), dtype=int)
    
    # Zone 1: Confident accept (low default probability)
    decisions[y_proba < t1] = 0
    
    # Zone 3: Confident reject (high default probability)
    decisions[y_proba > t2] = 1
    
    # Zone 2: Uncertain (manual review)
    decisions[(y_proba >= t1) & (y_proba <= t2)] = 2
    
    return decisions

def optimize_selective_thresholds(y_true, y_proba, 
                                  max_manual_review_rate=0.20,
                                  fn_cost=2.5, fp_cost=1.0):
    """
    Optimize t1 and t2 thresholds for selective prediction
    
    Objectives:
    1. Minimize cost (FN weighted higher)
    2. Keep manual review rate below max_manual_review_rate
    3. Maintain good FP/FN balance
    """
    best_cost = np.inf
    best_t1 = 0.3
    best_t2 = 0.5
    best_metrics = None
    
    # Search grid
    t1_range = np.arange(0.20, 0.45, 0.05)
    t2_range = np.arange(0.45, 0.70, 0.05)
    
    results = []
    
    for t1 in t1_range:
        for t2 in t2_range:
            if t1 >= t2:
                continue
            
            decisions = selective_prediction(y_proba, t1, t2)
            
            # Calculate metrics for automated decisions only
            auto_mask = decisions != 2
            manual_mask = decisions == 2
            
            if auto_mask.sum() == 0:
                continue
            
            y_true_auto = y_true[auto_mask]
            y_pred_auto = decisions[auto_mask]
            
            # Confusion matrix for automated decisions
            cm = confusion_matrix(y_true_auto, y_pred_auto)
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
            else:
                continue
            
            # Calculate metrics
            manual_review_rate = manual_mask.sum() / len(y_proba)
            
            if manual_review_rate > max_manual_review_rate:
                continue
            
            cost = fn_cost * fn + fp_cost * fp
            fp_fn_ratio = fp / fn if fn > 0 else fp
            
            # Penalty for imbalance
            if fp_fn_ratio < 3:
                cost += 100
            elif fp_fn_ratio > 20:
                cost += 50
            
            results.append({
                't1': t1,
                't2': t2,
                'cost': cost,
                'fn': fn,
                'fp': fp,
                'fp_fn_ratio': fp_fn_ratio,
                'manual_review_rate': manual_review_rate,
                'auto_decisions': auto_mask.sum(),
                'tn': tn,
                'tp': tp
            })
            
            if cost < best_cost:
                best_cost = cost
                best_t1 = t1
                best_t2 = t2
                best_metrics = {
                    'cost': cost,
                    'fn': fn,
                    'fp': fp,
                    'tn': tn,
                    'tp': tp,
                    'fp_fn_ratio': fp_fn_ratio,
                    'manual_review_rate': manual_review_rate,
                    'auto_decisions': auto_mask.sum()
                }
    
    results_df = pd.DataFrame(results).sort_values('cost')
    
    return best_t1, best_t2, best_metrics, results_df

print("Selective prediction functions defined successfully")

# ---------------------------------------------------------
# PART 8: OPTIMIZE SELECTIVE THRESHOLDS ON VALIDATION SET
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 8: OPTIMIZING SELECTIVE PREDICTION THRESHOLDS")
print("="*80)

print("Searching for optimal t1 and t2 thresholds...")
print("Constraints:")
print("  - Maximum manual review rate: 20%")
print("  - FN cost weight: 2.5x")
print("  - FP cost weight: 1.0x")
print("  - Target FP/FN ratio: 3-20")

best_t1, best_t2, selective_metrics, selective_results = optimize_selective_thresholds(
    y_val.values,
    y_val_pred_proba,
    max_manual_review_rate=0.20,
    fn_cost=2.5,
    fp_cost=1.0
)

print("\n" + "-"*80)
print("OPTIMAL SELECTIVE PREDICTION THRESHOLDS")
print("-"*80)
print(f"t1 (Accept threshold): {best_t1:.2f}")
print(f"t2 (Reject threshold): {best_t2:.2f}")
print(f"\nZone Definitions:")
print(f"  Zone 1 (Auto Accept): p(default) < {best_t1:.2f}")
print(f"  Zone 2 (Manual Review): {best_t1:.2f} ≤ p(default) ≤ {best_t2:.2f}")
print(f"  Zone 3 (Auto Reject): p(default) > {best_t2:.2f}")

print(f"\nPerformance Metrics:")
print(f"  Manual Review Rate: {selective_metrics['manual_review_rate']:.2%}")
print(f"  Automated Decisions: {selective_metrics['auto_decisions']} / {len(y_val)}")
print(f"  False Negatives (FN): {selective_metrics['fn']}")
print(f"  False Positives (FP): {selective_metrics['fp']}")
print(f"  FP/FN Ratio: {selective_metrics['fp_fn_ratio']:.2f}")
print(f"  Total Cost: {selective_metrics['cost']:.2f}")
print(f"  True Negatives (TN): {selective_metrics['tn']}")
print(f"  True Positives (TP): {selective_metrics['tp']}")

print("\nTop 10 Threshold Combinations:")
display(selective_results.head(10))

# ---------------------------------------------------------
# PART 9: TEST SELECTIVE PREDICTION ON TEST SET
# ---------------------------------------------------------
print("\n" + "="*80)
print("PART 9: EVALUATING SELECTIVE PREDICTION ON TEST SET")
print("="*80)

# Apply selective prediction
test_decisions = selective_prediction(y_test_pred_proba, best_t1, best_t2)

# Analyze results
auto_accept_mask = test_decisions == 0
auto_reject_mask = test_decisions == 1
manual_review_mask = test_decisions == 2

print(f"\nTest Set Distribution:")
print(f"  Auto Accept: {auto_accept_mask.sum()} ({auto_accept_mask.mean():.2%})")
print(f"  Auto Reject: {auto_reject_mask.sum()} ({auto_reject_mask.mean():.2%})")
print(f"  Manual Review: {manual_review_mask.sum()} ({manual_review_mask.mean():.2%})")

# Evaluate automated decisions
auto_mask = test_decisions != 2
y_test_auto = y_test[auto_mask]
y_pred_auto = test_decisions[auto_mask]

if len(y_test_auto) > 0:
    cm_auto = confusion_matrix(y_test_auto, y_pred_auto)
    print(f"\nConfusion Matrix (Automated Decisions Only):")
    print(cm_auto)
    
    if cm_auto.shape == (2, 2):
        tn, fp, fn, tp = cm_auto.ravel()
        print(f"\nTN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
        print(f"FP/FN Ratio: {fp/fn if fn > 0 else fp:.2f}")
        print(f"Cost (2.5×FN + 1×FP): {2.5*fn + 1.0*fp:.2f}")
        
        # Compare with single-threshold approach
        print(f"\nComparison with Single Threshold ({best_threshold:.2f}):")
        y_test_single = (y_test_pred_proba >= best_threshold).astype(int)
        cm_single = confusion_matrix(y_test, y_test_single)
        tn_s, fp_s, fn_s, tp_s = cm_single.ravel()
        print(f"  Single Threshold - FN: {fn_s}, FP: {fp_s}, Cost: {2.5*fn_s + 1.0*fp_s:.2f}")
        print(f"  Selective Prediction - FN: {fn}, FP: {fp}, Cost: {2.5*fn + 1.0*fp:.2f}")
        print(f"  Cost Reduction: {((2.5*fn_s + 1.0*fp_s) - (2.5*fn + 1.0*fp)):.2f}")
        print(f"  FN Reduction: {fn_s - fn}")

# Analyze manual review zone
if manual_review_mask.sum() > 0:
    manual_review_actual = y_test[manual_review_mask]
    manual_review_proba = y_test_pred_proba[manual_review_mask]
    
    print(f"\nManual Review Zone Analysis:")
    print(f"  Total cases: {manual_review_mask.sum()}")
    print(f"  Actual defaults: {manual_review_actual.sum()} ({manual_review_actual.mean():.2%})")
    print(f"  Actual non-defaults: {(1-manual_review_actual).sum()} ({(1-manual_review_actual).mean():.2%})")
    print(f"  Mean probability: {manual_review_proba.mean():.4f}")
    print(f"  Probability range: [{manual_review_proba.min():.4f}, {manual_review_proba.max():.4f}]")
